 **Classification du Churn Bancaire — Approche XGBoost (Python + Snowflake ML Registry)**

## 1. Contexte
 
#### Problematique
Une banque souhaite **identifier en avance les clients susceptibles de partir** (churn).  
Detecter ces clients permet de lancer des actions de retention ciblees avant qu'il ne soit trop tard.
 
### Pourquoi XGBoost ?
XGBoost (Extreme Gradient Boosting) est particulierement adapte a ce cas d'usage pour plusieurs raisons :
 
| Critere | XGBoost | Regression logistique | Random Forest |
|---|---|---|---|
| Gestion du desequilibre de classes | Excellent | Limite | Possible |
| Performance sur donnees tabulaires | Excellent | Moyen | Bon |
| Importance des features | Native | Non | Possible |
| Integration Snowflake ML Registry | Supporte | Supporte | Supporte |
| Temps d'entrainement | Rapide | Tres rapide | Plus lent |
 
XGBoost construit des arbres de decision en sequence, chaque arbre corrigeant les erreurs du precedent.  
C'est cette correction iterative qui lui donne un avantage sur des datasets de taille moderee comme le notre.
 
### Objectif
- **Variable cible :** `A_QUITTE` (0 = client stable, 1 = client parti)
- **Dataset :** 210 clients, 7 features comportementales et financieres

In [ ]:
%%sql -r dataframe_2
INSERT INTO CLIENTS VALUES
('C001', 'Marie Dupont',    45, 'Paris',    '2018-03-15', 2500.0,  1),
('C002', 'Jean Martin',     32, 'Lyon',     '2020-06-01', 800.0,   0),
('C003', 'Sophie Bernard',  28, 'Paris',    '2019-11-20', 150.0,   1),
('C004', 'Paul Leroy',      55, 'Bordeaux', '2017-01-10', 5200.0,  0),
('C005', 'Emma Moreau',     23, 'Lille',    '2021-09-05', 300.0,   1),
('C006', 'Lucas Simon',     41, 'Paris',    '2016-07-22', 3800.0,  0),
('C007', 'Camille Laurent', 36, 'Nantes',   '2019-04-18', 1200.0,  0),
('C008', 'Thomas Petit',    29, 'Lyon',     '2022-01-30', 90.0,    1),
('C009', 'Julie Roux',      48, 'Marseille','2015-12-05', 6100.0,  0),
('C010', 'Antoine Blanc',   33, 'Paris',    '2020-08-14', 450.0,   1);

INSERT INTO TRANSACTIONS VALUES
('T001', 'C001', '2024-01-05',  -120.0,  'ACHAT'),
('T002', 'C001', '2024-01-15',  -45.0,   'ACHAT'),
('T003', 'C001', '2024-02-01',  2500.0,  'DEPOT'),
('T004', 'C002', '2024-01-10',  -800.0,  'VIREMENT'),
('T005', 'C002', '2024-01-20',  -200.0,  'RETRAIT'),
('T006', 'C002', '2024-02-10',  -150.0,  'ACHAT'),
('T007', 'C003', '2024-01-08',  -30.0,   'ACHAT'),
('T008', 'C004', '2024-01-12',  5000.0,  'DEPOT'),
('T009', 'C004', '2024-01-25',  -1200.0, 'VIREMENT'),
('T010', 'C004', '2024-02-05',  -300.0,  'ACHAT'),
('T011', 'C005', '2024-01-03',  -25.0,   'ACHAT'),
('T012', 'C006', '2024-01-18',  -500.0,  'VIREMENT'),
('T013', 'C006', '2024-02-08',  3800.0,  'DEPOT'),
('T014', 'C007', '2024-01-22',  -180.0,  'ACHAT'),
('T015', 'C007', '2024-02-12',  -90.0,   'RETRAIT'),
('T016', 'C008', '2024-01-06',  -15.0,   'ACHAT'),
('T017', 'C009', '2024-01-14',  6000.0,  'DEPOT'),
('T018', 'C009', '2024-02-03',  -2000.0, 'VIREMENT'),
('T019', 'C010', '2024-01-09',  -60.0,   'ACHAT'),
('T020', 'C010', '2024-01-28',  -40.0,   'ACHAT');

select * from clients;
select * from transactions;

-- On crée la feature 

use schema ml_churn_project.feature_store;
create or replace view feature_store.client_features as
select
    c.client_id,
    datediff('day', c.date_inscription, current_date())     as anciennete_jours,
    count(t.transaction_id)                                  as nb_transactions,
    round(avg(t.montant), 2)                                 as montant_moyen,
    sum(case when t.type_transacation = 'RETRAIT' 
             then 1 else 0 end)                              as nb_retraits,
    round(
        sum(case when t.type_transacation = 'RETRAIT' 
                 then 1 else 0 end) / nullif(count(*), 0)
    , 2)                                                     as ratio_retraits,
    c.solde_moyen                                            as solde_moyen,
    c.age                                                    as age,
    c.a_quitte                                               as a_quitte
from ml_churn_project.data_raw.clients c
left join ml_churn_project.data_raw.transactions t
    on c.client_id = t.client_id
group by
    c.client_id,
    c.date_inscription,
    c.solde_moyen,
    c.age,
    c.a_quitte;

    select * from feature_store.client_features;
  use database ml_churn_project;
use schema data_raw;
use warehouse ml_wh;

-- Générer 200 nouveaux clients 
insert into clients
select
    'C' || lpad(row_number() over (order by seq4()) + 10, 4, '0') as client_id,
    
    case when uniform(1,2,random()) = 1 
         then 'Client_M_' else 'Client_F_' 
    end || (row_number() over (order by seq4()) + 10)              as nom,
    
    uniform(18, 70, random())                                       as age,
    
    case uniform(1,5,random())
        when 1 then 'Paris'
        when 2 then 'Lyon'
        when 3 then 'Marseille'
        when 4 then 'Bordeaux'
        else 'Lille'
    end                                                             as ville,
    
    dateadd('day', -uniform(365, 3650, random()), current_date())   as date_inscription,
    
    case when uniform(1,2,random()) = 1
         then uniform(50, 800, random())::float
         else uniform(500, 8000, random())::float
    end                                                             as solde_moyen,

    case when solde_moyen < 500 and uniform(1,10,random()) > 3 then 1
         when solde_moyen < 200 then 1
         when uniform(1,10,random()) > 8 then 1
         else 0
    end                                                             as a_quitte

from table(generator(rowcount => 200));

-- Générer 1000 transactions pour tous les clients
insert into transactions
select
    'T' || lpad(row_number() over (order by seq4()), 6, '0') as transaction_id,
    
    c.client_id,
    
    -- date de transaction de la transaction
    dateadd('day', -uniform(1, 365, random()), current_date()) as date_transaction,
    
    
    case 
        when uniform(1,4,random()) = 1 then  uniform(100, 5000, random())::float   -- DEPOT
        when uniform(1,4,random()) = 2 then -uniform(50,  2000, random())::float   -- VIREMENT
        when uniform(1,4,random()) = 3 then -uniform(10,  500,  random())::float   -- ACHAT
        else                                 -uniform(20,  800,  random())::float   -- RETRAIT
    end as montant,
    
    -- type : clients à risque font plus de retraits
    case
        when c.a_quitte = 1 then
            case uniform(1,4,random())
                when 1 then 'RETRAIT'
                when 2 then 'RETRAIT'
                when 3 then 'ACHAT'
                else        'VIREMENT'
            end
        else
            case uniform(1,4,random())
                when 1 then 'DEPOT'
                when 2 then 'ACHAT'
                when 3 then 'VIREMENT'
                else        'RETRAIT'
            end
    end as type_transaction

from clients c,
     table(generator(rowcount => 5))  -- 5 transactions par client en moyenne

order by random();

select * from clients;
select * from transactions;

## 3. Methode 
 
#### Pipeline complet


#### Approche 1 : XGBoost (Python + Snowpark)
```
  
Snowflake (DATA_RAW)
        ↓
  Feature Store (VIEW)
        ↓    
                                                                                            
  Snowpark - Pandas                                                     
        ↓
  Train / Test Split (80/20)
        ↓
  XGBoost Classifier
        ↓
  Évaluation 
        ↓
  Snowflake ML Registry
        ↓
  Scoring sur tous les clients
```

#### Approche 2 : Fonctions Natives Snowflake

```
Snowflake (DATA_RAW)
     ↓
Feature Store (VIEW)
     ↓
SNOWFLAKE.ML.CLASSIFICATION (input_data=CLIENT_FEATURES, target_colname=A_QUITTE)
     ↓
Évaluation (métriques automatiques)
    ↓
Modèle sauvegardé dans ML Registry (CHURN_PREDICTOR_NATIVE)
    ↓
Scoring sur tous les clients

```

## 4. Implementation

### 4.1 Approche 1 : XGBoost (Python)


In [ ]:
from snowflake.snowpark.context import get_active_session
from sklearn.model_selection import train_test_split
import xgboost as xgb

session = get_active_session()
session.sql("use database ml_churn_project").collect()
session.sql("use schema ml_models").collect()
session.sql("use warehouse ml_wh").collect()

print("Base :", session.get_current_database())
print("Schema :", session.get_current_schema())
print("Warehouse :", session.get_current_warehouse())

In [ ]:
df = session.table("ml_churn_project.feature_store.client_features")
df_pandas = df.to_pandas()
print(f"Nombre de clients : {len(df_pandas)}")
print(f"Repartition : {df_pandas['A_QUITTE'].value_counts().to_dict()}")
df_pandas.head(5)

### Hyperparamètres XGBoost retenus
| Paramètre |Valeur | Justification |
|---|---|---|
| `n_estimators` | 100| Bon compromis vitesse/performance sur 210 lignes |
| `max_depth` | 6 | Évite le surapprentissage sur un petit dataset ( Ne change plus à partir de profondeur 6 |
| `learning_rate`| 0.1 | Standard pour XGBoost |
| `eval_metric` | logloss | Adapté à la classification binaire |


In [ ]:
import pandas as pd

features = [
    "ANCIENNETE_JOURS",
    "NB_TRANSACTIONS", 
    "MONTANT_MOYEN",
    "NB_RETRAITS",
    "RATIO_RETRAITS",
    "SOLDE_MOYEN",
    "AGE"
]
X = df_pandas[features]
y = df_pandas["A_QUITTE"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print(f"Train : {len(X_train)} clients | Test : {len(X_test)} clients")

modele = xgb.XGBClassifier(
    n_estimators=100,      
    max_depth=6,           
    learning_rate=0.1,     
    random_state=42,
    eval_metric="logloss"
)
modele.fit(X_train, y_train)

score = modele.score(X_test, y_test)
print(f"Accuracy sur le test : {score:.1%}")

importance = pd.DataFrame({
    "feature": features,
    "importance": modele.feature_importances_
}).sort_values("importance", ascending=False)
print("\nImportance des features :")
print(importance.to_string(index=False))

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="ML_CHURN_PROJECT",
    schema_name="ML_MODELS"
)

try:
    registry.delete_model("CHURN_PREDICTOR")
    print("Ancien modele supprime")
except:
    print("Pas d'ancien modele a supprimer")

model_ref = registry.log_model(
    model=modele,
    model_name="CHURN_PREDICTOR",
    version_name="V1",
    sample_input_data=X_train,
    target_platforms=["WAREHOUSE"],
    comment="Modele XGBoost prediction churn bancaire"
)
print("Modele sauvegarde dans le registry")

df_all = session.table("ML_CHURN_PROJECT.FEATURE_STORE.CLIENT_FEATURES")
predictions = model_ref.run(df_all, function_name="predict")
df_predictions = predictions.to_pandas()

df_final = df_predictions[[
    "CLIENT_ID", "SOLDE_MOYEN", "NB_RETRAITS", "A_QUITTE", "output_feature_0"
]].copy()
df_final.columns = ["CLIENT_ID", "SOLDE_MOYEN", "NB_RETRAITS", "REEL", "PREDICTION"]
df_final["RESULTAT"] = df_final.apply(
    lambda r: "Correct" if r["REEL"] == r["PREDICTION"] else "Erreur", axis=1
)

print(f"\nTotal clients scores : {len(df_final)}")
print(f"Clients a risque (1) : {(df_final['PREDICTION'] == 1).sum()}")
print(f"Clients stables  (0) : {(df_final['PREDICTION'] == 0).sum()}")
df_final.head(10)

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = modele.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print("Matrice de Confusion")
print(f"  Vrais Negatifs  (reste -> predit reste) : {cm[0][0]}")
print(f"  Faux Positifs   (reste -> predit parti) : {cm[0][1]}")
print(f"  Faux Negatifs   (parti -> predit reste) : {cm[1][0]}")
print(f"  Vrais Positifs  (parti -> predit parti) : {cm[1][1]}")

print("\nRapport de Classification")
print(classification_report(y_test, y_pred, target_names=["Reste (0)", "Parti (1)"]))

In [ ]:
%%sql -r dataframe_4
-- This is your Cortex Project.
-----------------------------------------------------------
-- SETUP
-----------------------------------------------------------
use role ACCOUNTADMIN;
use warehouse ML_WH;
use database ML_CHURN_PROJECT;
use schema FEATURE_STORE;

-- Inspect the first 10 rows of your training data. This is the data we'll
-- use to create your model.
select * from CLIENT_FEATURES limit 10;

-- Inspect the first 10 rows of your prediction data. This is the data the model
-- will use to generate predictions.
select * from CLIENT_FEATURES limit 10;

-----------------------------------------------------------
-- CREATE PREDICTIONS
-----------------------------------------------------------
-- Create your model.
CREATE OR REPLACE SNOWFLAKE.ML.CLASSIFICATION my_model(
    INPUT_DATA => SYSTEM$REFERENCE('VIEW', 'CLIENT_FEATURES'),
    TARGET_COLNAME => 'A_QUITTE',
    CONFIG_OBJECT => { 'ON_ERROR': 'SKIP' }
);

-- Inspect your logs to ensure training completed successfully. 
CALL my_model!SHOW_TRAINING_LOGS();

-- Generate predictions as new columns in to your prediction table.
CREATE OR REPLACE TABLE My_classification_2026_05_04 AS SELECT
    *, 
    my_model!PREDICT(
        OBJECT_CONSTRUCT(*),
        -- This option alows the prediction process to complete even if individual rows must be skipped.
        {'ON_ERROR': 'SKIP'}
    ) as predictions
from CLIENT_FEATURES;

-- View your predictions.
SELECT * FROM My_classification_2026_05_04;

-- Parse the prediction results into separate columns. 
-- Note: This is a just an example. Be sure to update this to reflect 
-- the classes in your dataset.
SELECT * EXCLUDE predictions,
        predictions:class AS class,
        round(predictions['probability'][class], 3) as probability
FROM My_classification_2026_05_04;

-----------------------------------------------------------
-- INSPECT RESULTS
-----------------------------------------------------------

-- Inspect your model's evaluation metrics.
CALL my_model!SHOW_EVALUATION_METRICS();
CALL my_model!SHOW_GLOBAL_EVALUATION_METRICS();
CALL my_model!SHOW_CONFUSION_MATRIX();

-- Inspect the relative importance of your features, including auto-generated features.  
CALL my_model!SHOW_FEATURE_IMPORTANCE();

SELECT ML_CHURN_PROJECT.FEATURE_STORE.MODELE_CHURN!PREDICT(
    INPUT_DATA => OBJECT_CONSTRUCT(
        'CLIENT_ID' , 'C0123',
        'AGE', 45,
        'ANCIENNETE_JOURS', 365,
        'NB_TRANSACTIONS', 10,
        'MONTANT_MOYEN', -200,
        'NB_RETRAITS', 7,
        'RATIO_RETRAITS', 0.70,
        'SOLDE_MOYEN', 150
    )
) AS PREDICTION;

## 5. Resultats - Comparaison des deux approches

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

print("APPROCHE 1 : XGBoost (Python) - jeu de TEST (20%)")

y_pred_xgb = modele.predict(X_test)
report_xgb = classification_report(y_test, y_pred_xgb, output_dict=True)
print(f"Accuracy : {accuracy_score(y_test, y_pred_xgb):.1%}")
print(classification_report(y_test, y_pred_xgb, target_names=["Reste (0)", "Parti (1)"]))

print("APPROCHE 2 : Classification Native Snowflake")
print("(Metriques issues de SHOW_EVALUATION_METRICS - split interne)")


native_precision_0 = 0.621
native_recall_0 = 0.818
native_f1_0 = 0.706
native_support_0 = 22
native_precision_1 = 0.692
native_recall_1 = 0.450
native_f1_1 = 0.545
native_support_1 = 20
native_accuracy = (native_recall_0 * native_support_0 + native_recall_1 * native_support_1) / (native_support_0 + native_support_1)

print(f"Accuracy (approx.) : {native_accuracy:.1%}")
print(f"{'':>14}precision    recall  f1-score   support")
print(f"   Reste (0)      {native_precision_0:.2f}      {native_recall_0:.2f}      {native_f1_0:.2f}        {native_support_0}")
print(f"   Parti (1)      {native_precision_1:.2f}      {native_recall_1:.2f}      {native_f1_1:.2f}        {native_support_1}")

print("\nTABLEAU COMPARATIF - PERFORMANCE")


comparaison = pd.DataFrame({
    "Metrique": ["Accuracy", "Precision (Parti)", "Recall (Parti)", "F1 (Parti)"],
    "XGBoost (Python)": [
        f"{accuracy_score(y_test, y_pred_xgb):.1%}",
        f"{report_xgb['1']['precision']:.1%}",
        f"{report_xgb['1']['recall']:.1%}",
        f"{report_xgb['1']['f1-score']:.1%}"
    ],
    "Classification Native": [
        f"{native_accuracy:.1%}",
        f"{native_precision_1:.1%}",
        f"{native_recall_1:.1%}",
        f"{native_f1_1:.1%}"
    ]
})
print(comparaison.to_string(index=False))

print("\n\nTABLEAU COMPARATIF - COUT")


cout = pd.DataFrame({
    "Poste": [
        "Compute (training)",
        "Compute (inference)",
        "Stockage modele",
        "Estimation totale"
    ],
    "XGBoost (Python)": [
        "Notebook Service XS : 1 credit/h",
        "Warehouse pour predict : 1 credit",
        "Registry : negligeable",
        "2-3 credits"
    ],
    "Classification Native": [
        "Warehouse ML_WH : 5-10 credits",
        "Warehouse pour predict : 1 credit",
        "Modele interne : negligeable",
        "6-11 credits"
    ]
})
print(cout.to_string(index=False))

print("\nNotes :")
print("- La classification native utilise son propre split interne.")
print("- XGBoost utilise un split 80/20 (random_state=42).")
print("- Le cout reel depend de la taille du warehouse et du temps d'execution.")

## 6. Options d'optimisation

| Piste | Description | Impact attendu |
|-------|-------------|----------------|
| **Augmenter les données** | Passer de 210 à 1000+ clients | Améliore la généralisation |
| **Feature engineering** | Ajouter tendance du solde, fréquence mensuelle, délai dernière transaction | Capture plus de signaux |
| **Hyperparameter tuning** | GridSearch sur max_depth, learning_rate, n_estimators | +2-5% de F1 |
| **SMOTE** | Suréchantillonnage de la classe minoritaire (churn=1) | Améliore le recall |
| **Validation croisée** | 5-fold CV au lieu de train/test simple | Estimation plus fiable |

## 7. Pour aller plus loin

| Axe | Description |
|-----|-------------|
| **Déploiement** | Utiliser le modèle via `PREDICT()` pour scorer les nouveaux clients en temps réel |
| **Monitoring** | Suivre la performance du modèle dans le temps (drift detection) |
| **Seuil de décision** | Ajuster le seuil de probabilité (ex: 0.3 au lieu de 0.5) pour détecter plus de churners |
| **Segmentation** | Combiner les prédictions avec un clustering pour personnaliser les actions de rétention |
| **Explicabilité** | Utiliser SHAP values pour expliquer chaque prédiction individuellement |
| **Automatisation** | Planifier un re-training périodique avec Snowflake Tasks |